# Demo

This notebook demonstrates how to use the `stung` library to profile collinearity breaks across the two haplotypes in a haplotype-resolved diploid genome assembly.

In [ ]:
# imports

from stung.bumble import *
from stung.utils import *
from stung.stung import *

start by defining input file paths:

In [ ]:
genome_file_path = "/path/to/genome.fasta"

# both gff and gtf file formats are accepted
annotation_file_paths = [
    "/path/to/annotation.hap1.gff",
    "/path/to/annotation.hap2.gff"
]

temp_dir = "/path/to/temp_dir"
out_dir = "/path/to/output_dir"

then initialize a `Bumble` instance and build the initial 2d match matrix recording gene name / symbol matches:

In [ ]:
bmbl = Bumble(
    genome_fp = genome_file_path,
    ann_fps = annotation_file_paths,
    temp_dir = temp_dir,
    out_dir = out_dir
)

In [ ]:
mat, n = bmbl.build_2d_matrix()

a 2d matrix can be saved and visualized as a dot plot (interactive / static):

In [ ]:
# saving preliminary 2d match matrix

save_2d_matrix(mat, os.path.join(out_dir, 'mmat.pre.tsv'))

In [ ]:
# generates an interactive plot by default
# percent identity values not available since we have not called buzz() function yet

plot_2d_matrix(
    mat = mat,
    pident_mat = None,
    x_gene_order = bmbl.x_gene_order,
    y_gene_order = bmbl.y_gene_order,
    n = n
)

the main operation of the `stung` library is encapsulated in this `buzz()` function that first identifies colinear blocks, idenitfying breaks in between them (i.e., stungs), and then extending the colinear boundaries based on fast pairwise nucleotide-level alignments (using [a*pa2](https://github.com/RagnarGrootKoerkamp/astar-pairwise-aligner)):

In [ ]:
_, _, _ = buzz(
    mat = mat,
    bmbl = bmbl,
    n = n,
    out_dir = out_dir,
    verbose = True
)

after this compute-intensive step of this analysis, we should save the match matrix, percent identity matrix, and visualize the matches again:

In [ ]:
# also save the match matrix which probably changed based on all the alignment results
save_2d_matrix(mat, os.path.join(out_dir, 'mmat.post.tsv'))

# save percent identities
# percent identity values are only available for select pairs of genes around stungs
save_2d_matrix(bmbl.pident_mat, os.path.join(out_dir, 'pident.tsv'))

In [ ]:
# percent identities are now available
plot_2d_matrix(
    mat = mat,
    pident_mat = bmbl.pident_mat,
    x_gene_order = bmbl.x_gene_order,
    y_gene_order = bmbl.y_gene_order,
    n = n
)

In [ ]:
# you can also visualize the matrix as a static plot (and save it)
# here we only select a small region of the match matrix

# IMPORTANT : replace the x/y start end values below with the region you'd to visualize
x_start = -1
x_end = -1
y_start = -1
y_end = -1
dot_s = 50

plot_2d_matrix(
    mat = mat[y_start:y_end, x_start:x_end],
    pident_mat = None,
    x_gene_order = bmbl.x_gene_order,
    y_gene_order = bmbl.y_gene_order,
    n = n,
    is_interactive = False,
    dot_s = dot_s,
    save = True,
    out_file_path = os.path.join(out_dir, f'x_{x_start}:{x_end}_y_{y_start}:{y_end}.png'),
    x_offset = x_start,
    y_offset = y_start
)